## Setup

In [ ]:
!git clone https://github.com/realBarry123/audio-embed-experiments
%cd audio-embed-experiments

In [ ]:
!source /venv/main/bin/activate

In [ ]:
!/venv/main/bin/pip install -r requirements.txt
!/venv/main/bin/pip install -e .

In [ ]:
import huggingface_hub
huggingface_hub.notebook_login()

In [ ]:
import wandb
wandb.login()

In [ ]:
!mkdir experiments/sae/results

## Training

In [ ]:
!/venv/main/bin/python experiments/sae/sae.py

In [ ]:
!/venv/main/bin/python experiments/sae/probe.py

## Reloading

In [ ]:
!git pull

In [ ]:
import importlib
import audembed
importlib.reload(audembed)
importlib.reload(audembed.data)

## Testing

### SAE

In [ ]:
!/venv/main/bin/python experiments/sae/test_sae.py

In [ ]:
%matplotlib inline
import torch
import matplotlib.pyplot as plt
from audembed import data

DIMS = 64
latent = torch.load("experiments/sae/results/orchset_sae_latent.pt")
plt.rcParams['figure.figsize'] = [latent.shape[1] * 0.2, DIMS * 0.15]
data.plot_heatmap_2d(
    latent[:DIMS],
    #data.sort_sae_latents(latent), 
    xlabel="t", ylabel="dim", 
    save_file="experiments/sae/results/orchset_sae_latent.png"
)

### Probe

In [ ]:
!/venv/main/bin/python experiments/sae/test_probe.py

In [ ]:
%matplotlib inline
import torch
import matplotlib.pyplot as plt
from audembed import data
bins = torch.arange(0, 128, 1)
r2 = torch.load("experiments/sae/results/probe_r2.pt")

plt.scatter(x=bins, y=r2)
plt.xlabel("bins")
plt.ylabel("R^2")
plt.show()

In [ ]:

import torch
state_dict, configs, start_epoch = torch.load("experiments/sae/models/probe.pt", map_location=torch.device('cpu'))
weight = state_dict["linear.weight"]
print(weight.shape)
plt.rcParams['figure.figsize'] = [10, 10]

from audembed import data
import matplotlib.colors as mcolors
cmap = mcolors.LinearSegmentedColormap.from_list(
    "orange_white_blue", ["orange", "white", "blue"]
)
data.plot_heatmap_2d(
    weight, 
    xlabel="SAE latent", 
    ylabel="bins", 
    cmap=cmap, 
    vmin=-weight.max(), 
    vmax=weight.max()
)


In [ ]:
import torch
state_dict, configs, start_epoch = torch.load("experiments/sae/models/probe.pt", map_location=torch.device('cpu'))
weight = state_dict["linear.weight"]

plt.plot(weight.norm(dim=1))
plt.xlabel("bins")
plt.ylabel("norm weight")
plt.show()